# Scratchpad and Working Memory

Long-horizon reasoning problems exceed what a model can hold in its reasoning chain before generating the final answer. Scratchpad management — using the context window as structured working memory — is the practical solution for extended multi-step computation.

## The Context-as-Memory Paradigm

Transformer models have no persistent memory outside the context window. For extended reasoning, the context window serves as both the input and the working memory. This creates a constraint: complex problems that require tracking many intermediate values can exhaust the effective attention capacity.

Scratchpad methods make this explicit and structured:
- Write intermediate results to a designated section of the context
- Organize working memory with headers, tables, or key-value notation
- Periodically compress and summarize older scratchpad content to free space
- Retrieve from scratchpad when needed rather than recomputing

In [ ]:
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class ScratchpadEntry:
    key: str
    value: str
    step: int
    compressed: bool = False

class Scratchpad:
    def __init__(self, max_entries: int = 20, compress_threshold: int = 15):
        self.entries: list = []
        self.max_entries = max_entries
        self.compress_threshold = compress_threshold
        self.step = 0

    def write(self, key: str, value: str):
        self.step += 1
        if len(self.entries) >= self.compress_threshold:
            self._compress_old_entries()
        self.entries.append(ScratchpadEntry(key=key, value=value, step=self.step))

    def read(self, key: str) -> str:
        for entry in reversed(self.entries):
            if entry.key == key:
                return entry.value
        return ''

    def _compress_old_entries(self):
        # Keep recent entries, summarize old ones
        old = self.entries[:5]
        compressed_value = 'Compressed: ' + '; '.join(f'{e.key}={e.value[:20]}' for e in old)
        self.entries = [ScratchpadEntry(key='summary', value=compressed_value,
                                         step=old[0].step, compressed=True)]
        self.entries.extend(self.entries[5:])  # keep recent
        self.entries = [ScratchpadEntry(key='summary', value=compressed_value,
                                         step=old[-1].step, compressed=True)] + self.entries[-10:]

    def render(self) -> str:
        lines = ['=== SCRATCHPAD ===']
        for e in self.entries:
            prefix = '[compressed]' if e.compressed else f'[step {e.step}]'
            lines.append(f'{prefix} {e.key}: {e.value}')
        return '\n'.join(lines)

    def token_count(self) -> int:
        return sum(len(e.key.split()) + len(e.value.split()) for e in self.entries)

# Simulate multi-step calculation with scratchpad
pad = Scratchpad(max_entries=20)
pad.write('problem', 'Find the compound interest on $1000 at 5% for 3 years')
pad.write('principal', '1000')
pad.write('rate', '0.05')
pad.write('years', '3')
pad.write('year1_interest', '1000 * 0.05 = 50')
pad.write('year1_balance', '1000 + 50 = 1050')
pad.write('year2_interest', '1050 * 0.05 = 52.5')
pad.write('year2_balance', '1050 + 52.5 = 1102.5')
pad.write('year3_interest', '1102.5 * 0.05 = 55.125')
pad.write('year3_balance', '1102.5 + 55.125 = 1157.625')
pad.write('answer', 'Total interest = 1157.625 - 1000 = 157.625')

print(pad.render())
print(f'\nToken estimate: {pad.token_count()}')
print(f'\nReading back: principal = {pad.read("principal")}')
print(f'Answer: {pad.read("answer")}')

## Context Compression Strategies

When reasoning chains grow long, compression becomes necessary:

**Summarization**: periodically ask the model to summarize completed reasoning steps into a compact form. Loses detail but maintains essential conclusions.

**Key-value extraction**: instead of raw reasoning text, maintain a structured dictionary of derived facts. Easier to query and compress.

**Hierarchical notes**: organize scratchpad into levels — raw computation at the bottom, derived conclusions at the top. Prune raw computation when conclusion is confirmed.

**Memory chunking**: split long problems into independent sub-problems, solve each with a fresh context, and combine results in a final synthesis pass.

In [ ]:
def compress_reasoning(steps: list, model_fn: Callable) -> str:
    combined = '\n'.join(f'Step {i+1}: {s}' for i, s in enumerate(steps))
    prompt = f'Summarize the following reasoning steps into 2-3 key conclusions:\n{combined}\n\nSummary:'
    return model_fn(prompt)

def scratchpad_pipeline(problem: str, steps_fn: Callable, model_fn: Callable,
                         max_active_steps: int = 5) -> dict:
    pad = Scratchpad(max_entries=30)
    pad.write('problem', problem)
    raw_steps = steps_fn(problem)
    active_steps = []
    compressed_summaries = []
    for i, step in enumerate(raw_steps):
        active_steps.append(step)
        pad.write(f'step_{i+1}', step)
        if len(active_steps) >= max_active_steps:
            summary = compress_reasoning(active_steps, model_fn)
            compressed_summaries.append(summary)
            pad.write(f'summary_{len(compressed_summaries)}', summary)
            active_steps = []
    return {'pad': pad, 'summaries': compressed_summaries,
            'total_steps': len(raw_steps), 'compressions': len(compressed_summaries)}

def mock_compress(prompt):
    return 'Key conclusions: computed intermediate values and verified constraints'

steps = [
    '1000 * 0.05 = 50 (year 1 interest)',
    '1000 + 50 = 1050 (year 1 balance)',
    '1050 * 0.05 = 52.5 (year 2 interest)',
    '1050 + 52.5 = 1102.5 (year 2 balance)',
    '1102.5 * 0.05 = 55.125 (year 3 interest)',
    '1102.5 + 55.125 = 1157.625 (final balance)',
    '1157.625 - 1000 = 157.625 (total interest)',
]
result = scratchpad_pipeline('Compound interest $1000 5% 3 years', lambda p: steps, mock_compress, max_active_steps=4)
print(f'Total steps: {result["total_steps"]}')
print(f'Compressions performed: {result["compressions"]}')
print(f'Summaries: {result["summaries"]}')
print(f'Final scratchpad tokens: {result["pad"].token_count()}')

## Extended Context vs Scratchpad

Long-context models (128K+ tokens) reduce the urgency of scratchpad management but do not eliminate it:
- Attention cost scales quadratically with context length — very long scratchpads are expensive
- Models struggle with retrieving specific facts buried in very long contexts (the 'lost in the middle' problem)
- Structured scratchpad with clear keys is more reliable than an unstructured chain of 50+ reasoning steps

Best practice: structure the working memory even when context length is not a bottleneck. Clear organization helps the model navigate its own intermediate results.